# INF2006 Task 3: Weekly volatility prediction (Spark ML)

**Question:** How volatile will a stock be next week (standard deviation of its daily log returns), and can predicted volatility be turned into risk-aware weights?
\
**Horizon:** one week.
\
**Features:** the Task 2 weekly features (past volatility over 1, 4, 12 and 52 weeks, the completed week's range and return, relative volume), all known on the anchor morning.
\
**Models compared:** Linear Regression, Random Forest, GBT.
\
**Baseline:** the stock's volatility over the previous 12 weeks (`vol_12w`).
\
**Output use:** weight = 1 / predicted volatility (floored), normalised to sum to 1 within each week.
\
**Optional Part B:** Model B adds Wikinews `news_count` (set `NEWS_TABLE`).

Run order: 0 Config -> 1 Load Task 2 weekly tables -> 2 Baselines -> 3 Training and Evaluating models -> 4 Freeze + test (4b optional Model B) -> 5 Save model (5b weights) -> 6 Evidence
\
**Section 7 (reload check) must be run in a fresh session** (restart the kernel, run section 0 only, then section 7).


## 0. Config, imports, helpers

In [ ]:
import json, sys, time, datetime
from pathlib import Path
from contextlib import contextmanager
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ---------- EDIT THESE (the only cell that needs to be changed between local and EMR) ----------
SEED = 42
TRAIN_TABLE   = "weekly_train"                   # Hive tables registered by Task 2 (task2/hive/01_register.sql): one row per stock per week
VAL_TABLE     = "weekly_validation"
TEST_TABLE    = "weekly_test"
WEEKLY_DIR    = "task2_outputs"                  # Leave None on EMR (reads the Hive tables).
OUT_BASE      = Path("task3_outputs").resolve().as_uri()    # EMR: "hdfs:///user/hadoop/inf2006/task3"
NEWS_TABLE    = None                             # OPTIONAL Part B: Hive table of Wikinews article revisions (one row per revision). Leave None to skip Model B.
NEWS_DATE_COL = "revision_date"                  # date/timestamp column of that table (when the revision was made)
EXPECTED_SPARK_PREFIX = "3.5"                    # EMR 7.10 ships Spark 3.5.x (confirm on the cluster: spark-submit --version).
STRICT_VERSION_CHECK  = False                    # set True on EMR to stop the run if the Spark version is different

# -----------------------------------------------------------------------------------------------

FEATURES_PATH = OUT_BASE + "/weekly_features" # engineered features
MODEL_PATH    = OUT_BASE + "/model_best" # saved final model
REF_PRED_PATH = OUT_BASE + "/reference_test_predictions" # test predictions for reproducibility. Test against the re-loaded model.
WEIGHTS_PATH  = OUT_BASE + "/test_weights" # weights built from the test predictions
EVIDENCE_PATH = OUT_BASE + "/evidence" # JSON logs, metrics, timings
TOL = 1e-9                     # reload-check tolerance on predicted volatility
FLOOR_PCT = 0.05               # predicted volatility is floored at this percentile of the TRAINING weekly volatilities before taking 1 / vol

# Define the time period to have data used for testing, validating and testing.
SPLITS = {"train": ("2013-01-01", "2015-12-31"),
          "val":   ("2016-01-01", "2016-12-31"),
          "test":  ("2017-01-01", "2017-12-31")}

# List of feature columns from Task 2
FEATS = ["vol_1w", "vol_4w", "vol_12w", "vol_52w", "high_low_range", "weekly_return", "volume_vs_normal", "split_flag"]
# vol_1w/4w/12w/52w = past volatility over 1/4/12/52 weeks. vol_52w was already filled with the TRAINING median in Task 2.

# Weekly row counts reported by Task 2 (task2_outputs/reconciliation.json), used to check what was loaded.
TASK2_EXPECTED_ROWS = {"train": 66907, "val": 25310, "test": 25441}

# Starts Spark, or reuses the one EMR already gave you.
spark = SparkSession.builder.appName("inf2006_task3").getOrCreate()
print("Spark", spark.version)

# Spark version check: the final run is on EMR. A model saved with a different Spark version may not load there.
SPARK_VERSION_OK = spark.version.startswith(EXPECTED_SPARK_PREFIX)
if not SPARK_VERSION_OK:
    msg = f"Spark {spark.version} is not the expected {EXPECTED_SPARK_PREFIX}.x. A model saved here may not load on EMR, so save the FINAL model on EMR."
    if STRICT_VERSION_CHECK:
        raise RuntimeError(msg)
    print("WARNING:", msg)

# Timer helper to track performance later.
TIMINGS = {}
@contextmanager
def stage(name):
    t0 = time.time(); yield
    TIMINGS[name] = round(time.time() - t0, 2); print(f"[timing] {name}: {TIMINGS[name]}s")

# Helper function that saves any Python dictionary as a text file in HDFS (or local filesystem).
def save_json(obj, path):
    spark.createDataFrame([(json.dumps(obj, indent=2, default=str),)], ["json"]).coalesce(1).write.mode("overwrite").text(path)

# Helpers to look at HDFS / local folders through Hadoop's FileSystem API (work for hdfs:// and file:// paths).
def _hadoop_fs(path):
    jpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    return jpath, jpath.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())

def path_exists(path):
    try:
        jpath, fs = _hadoop_fs(path); return fs.exists(jpath)
    except Exception:
        return False

def path_bytes(path):
    try:
        jpath, fs = _hadoop_fs(path); return fs.getContentSummary(jpath).getLength()
    except Exception as e:
        return f"unavailable ({type(e).__name__})"

Spark 4.0.5


## 1. Load the Task 2 weekly tables (labels, features, split assignment)
Task 2 already cleaned the prices and built the weekly tables, so this notebook does not rebuild them. Each row is one stock for one week.

* **Source**: `weekly_train` (2013-2015), `weekly_validation` (2016) and `weekly_test` (2017), read from Hive on EMR (or from Task 2's CSV files when running locally).
* **Anchor** = last trading day of a Monday-Friday week (the earlier trading day if Friday is a holiday). Features are known on that anchor morning.
* **Target** (column `label`) = Task 2's `target_next_vol`: the sample standard deviation of the stock's daily log returns over next week, from this anchor day's close up to the day before the next anchor day. It is not annualised. That is only about 4 returns per week (3 in holiday weeks; Task 2 needs at least 2), so the target is noisy. `target_next_vol` on one row equals `vol_1w` on the next row, and no feature uses information from the target week.
* **Features** (built from completed weeks only, i.e. up to the day before the anchor day; the anchor day's own close is never used): `vol_1w`, `vol_4w`, `vol_12w`, `vol_52w` (sample standard deviation of all daily log returns pooled over the last 1, 4, 12 or 52 completed weeks, not an average of weekly volatilities), `high_low_range` (completed week's high-low range ÷ previous anchor open), `weekly_return` (anchor open ÷ previous anchor open − 1), `volume_vs_normal` (completed week's volume ÷ the stock's median weekly volume over the 12 weeks before), `split_flag` (1 if the weekly return is 30% or more in size). `ticker`, `anchor_date` and `next_anchor_date` are not features. `vol_52w` was already filled with the training median in Task 2, so no new fill value is fitted for it here.
* **Done in Task 2**: rejecting invalid or out-of-range daily rows (kept in `task2_outputs/rejected_rows.csv`), building the weekly rows, the chronological split, and the boundary rule (a week keeps its target only if the next anchor is inside the same split). Five tickers with under a year of history (APTV, BHF, BHGE, DWDP, DXC) are left out of the weekly tables.
* **Checked here**: loaded row counts against Task 2's reconciliation, null counts per feature, every anchor date inside its own period, and the boundary rule again (all expected to pass with no exclusions). Every row still gets a `status`, so any exclusion is counted and reported rather than silently dropped.
* **Optional Part B**: if `NEWS_TABLE` is set, `news_count` = number of Wikinews revisions dated in the 7 days *before* the anchor day (so it is known at prediction time).

In [ ]:
# Returns “is column c between date a and date b?”. Used later to check that every row falls inside its own period
# (train/val/test).
def in_range(c, a, b):
    return (F.col(c) >= F.to_date(F.lit(a))) & (F.col(c) <= F.to_date(F.lit(b)))

# Read one Task 2 weekly table: from Hive on EMR, or from Task 2's CSV files locally (WEEKLY_DIR).
# Renames ticker -> symbol and target_next_vol -> label (the name Spark ML expects for the thing being predicted).
def load_weekly(table, csv_name, period):
    df = spark.table(table) if WEEKLY_DIR is None else spark.read.csv(f"{WEEKLY_DIR}/{csv_name}", header=True, inferSchema=True)
    # split_flag may arrive as boolean, 0/1 or the text True/False, so turn it into 1.0 / 0.0 explicitly.
    flag = F.when(F.lower(F.col("split_flag").cast("string")).isin("true", "1", "1.0"), F.lit(1.0)).otherwise(F.lit(0.0))
    return df.select(F.col("ticker").alias("symbol"),
                     F.to_date(F.col("anchor_date")).alias("anchor_date"),
                     F.to_date(F.col("next_anchor_date")).alias("next_anchor_date"),
                     F.lit(period).alias("period"),
                     F.col("target_next_vol").cast("double").alias("label"),
                     flag.alias("split_flag"),
                     *[F.col(c).cast("double").alias(c) for c in FEATS if c != "split_flag"])

with stage("build_weekly_table"):
    # Stack the three Task 2 tables. The period comes from which table the row was read from (no random shuffling).
    wk = (load_weekly(TRAIN_TABLE, "weekly_train.csv", "train")
            .unionByName(load_weekly(VAL_TABLE, "weekly_validation.csv", "val"))
            .unionByName(load_weekly(TEST_TABLE, "weekly_test.csv", "test")))

    # ---- OPTIONAL Part B: weekly Wikinews count (only if NEWS_TABLE is set).
    # news_count = number of news revisions dated in the 7 days BEFORE the anchor day, so it is known at prediction time.
    if NEWS_TABLE is not None:
        anchors = wk.select("anchor_date").distinct()
        news = spark.table(NEWS_TABLE).select(F.to_date(F.col(NEWS_DATE_COL)).alias("news_date"))
        news_wk = (news.join(F.broadcast(anchors), (F.col("news_date") >= F.date_sub(F.col("anchor_date"), 7)) & (F.col("news_date") < F.col("anchor_date")))
                       .groupBy("anchor_date").agg(F.count("*").alias("news_count")))
        wk = wk.join(news_wk, "anchor_date", "left").withColumn("news_count", F.coalesce(F.col("news_count"), F.lit(0)).cast("double"))

    # ---- period + boundary rule + status
    # Task 2 already applied these rules. They are re-checked here, so any violation shows up in the exclusions table.
    # Find the first and last day of each period.
    period_start = (F.when(F.col("period") == "train", F.to_date(F.lit(SPLITS["train"][0])))
                     .when(F.col("period") == "val",   F.to_date(F.lit(SPLITS["val"][0])))
                     .when(F.col("period") == "test",  F.to_date(F.lit(SPLITS["test"][0]))))
    period_end = (F.when(F.col("period") == "train", F.to_date(F.lit(SPLITS["train"][1])))
                   .when(F.col("period") == "val",   F.to_date(F.lit(SPLITS["val"][1])))
                   .when(F.col("period") == "test",  F.to_date(F.lit(SPLITS["test"][1]))))

    wk = wk.withColumn("status",
            F.when(F.col("label").isNull(), "target_unavailable")  # <- target missing
             .when(F.col("vol_12w").isNull(), "baseline_unavailable")  # <- the 12-week baseline needs vol_12w, so model and baseline use the same rows
             .when((F.col("anchor_date") < period_start) | (F.col("anchor_date") > period_end), "outside_periods")  # <- anchor date not inside its own period
             .when(F.col("next_anchor_date").isNull() | (F.col("next_anchor_date") > period_end), "crosses_boundary")  # <- prediction window goes past the allowed period.
                                                                                                                       # The “next week” anchor date is missing or lies outside the allowed period.
                                                                                                                       # That means the label would require future data we’re not allowed to use.
             .otherwise("ok"))  # <- Safe to use

    keep = ["symbol", "anchor_date", "next_anchor_date", "period", "status", "label"] + FEATS + (["news_count"] if NEWS_TABLE is not None else [])
    if path_exists(FEATURES_PATH):
      print("WARNING: output path already exists and will be overwritten.")
    wk.select(*keep).write.mode("overwrite").parquet(FEATURES_PATH)

data = spark.read.parquet(FEATURES_PATH)
summary = data.groupBy("period", "status").count().orderBy("period", "status")
summary.show(50, truncate=False)           # <- row counts and exclusions per split (paste into the report)
cov = (data.filter("status = 'ok'").groupBy("period")
           .agg(F.min("anchor_date").alias("first_anchor"), F.max("anchor_date").alias("last_anchor"),
                F.count("*").alias("rows"), F.avg("label").alias("mean_vol"), F.stddev("label").alias("std_vol")).orderBy("period"))
cov.show(truncate=False)

# Reconcile with the Task 2 hand-off: the rows loaded here should equal the weekly counts Task 2 reported.
loaded_counts = {r["period"]: r["count"] for r in data.groupBy("period").count().collect()}
print("rows loaded:", loaded_counts, "| expected from Task 2:", TASK2_EXPECTED_ROWS, "| match:", loaded_counts == TASK2_EXPECTED_ROWS)

# Null check: Task 2 already filled vol_52w, so every feature should have 0 nulls in every period.
data.groupBy("period").agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in FEATS]).orderBy("period").show(truncate=False)

[timing] build_weekly_table: 8.27s
+------+------+-----+
|period|status|count|
+------+------+-----+
|test  |ok    |25441|
|train |ok    |66907|
|val   |ok    |25310|
+------+------+-----+

+------+------------+-----------+-----+--------------------+--------------------+
|period|first_anchor|last_anchor|rows |mean_vol            |std_vol             |
+------+------------+-----------+-----+--------------------+--------------------+
|test  |2017-01-06  |2017-12-22 |25441|0.011173448873773452|0.009364968276556557|
|train |2013-05-10  |2015-12-24 |66907|0.012599516195070264|0.009367644656350723|
|val   |2016-01-08  |2016-12-23 |25310|0.014090359028496932|0.011495710018665574|
+------+------------+-----------+-----+--------------------+--------------------+

rows loaded: {'train': 66907, 'test': 25441, 'val': 25310} | expected from Task 2: {'train': 66907, 'val': 25310, 'test': 25441} | match: True
+------+------+------+-------+-------+--------------+-------------+----------------+--------

## 2. Baseline
* **12-week volatility** (`vol_12w` from Task 2): predict that next week's volatility equals the stock's volatility over the previous 12 completed weeks (the standard deviation of all daily log returns in those weeks). Rows without it are excluded as `baseline_unavailable`, so the baseline is scored on exactly the same rows as the models (but Task 2 removed incomplete rows, so this shouldn't happen).

In [ ]:
ok = data.filter("status = 'ok'")
train, val, test = [ok.filter(F.col("period") == p).cache() for p in ["train", "val", "test"]]
print("rows  train/val/test:", train.count(), val.count(), test.count())

# Regression metrics: lower RMSE / MAE is better; R2 above 0 means better than predicting the mean of the evaluated set.
def metrics(df, pred="prediction"):
    ev = lambda m: RegressionEvaluator(labelCol="label", predictionCol=pred, metricName=m).evaluate(df)
    return {"rmse": round(ev("rmse"), 6), "mae": round(ev("mae"), 6), "r2": round(ev("r2"), 4)}

# Baseline model
# Predict that next week's volatility equals the volatility over the previous 12 weeks.
# uses vol_12w from Task 2: standard deviation of the daily log returns over the previous 12 completed weeks (known on the anchor morning).
def baseline_vol_12w(df):
    return df.withColumn("prediction", F.col("vol_12w"))

# The ML model must beat these baselines, otherwise it’s not useful.
BASELINES = {"vol_12w": baseline_vol_12w}

# For each baseline model, run it on the validation dataset, compute metrics, and store the results.
base_val = {k: metrics(f(val)) for k, f in BASELINES.items()}
print("baselines on VALIDATION:", base_val)

rows  train/val/test: 66907 25310 25441
baselines on VALIDATION: {'vol_12w': {'rmse': 0.011139, 'mae': 0.007658, 'r2': 0.0611}}


## 3. Training and Evaluating models
Pipeline: `Imputer(median)` -> `VectorAssembler` -> `StandardScaler` -> regressor. Imputer and scaler are learned from the training set only and applied unchanged to validation/test.

In [ ]:
# Every candidate model uses this pipeline.
# Imputer fills missing values (Task 2 already filled vol_52w from the training rows, so this is only a safeguard; it is fitted on training data only).
# Assembler combines features.
# Scaler normalizes them.
# clf is the model (Linear Regression, RF, GBT).
# feats is the list of feature columns (Model A = FEATS; the optional Model B adds news_count).
def make_pipeline(clf, feats=FEATS):
    imp_cols = [c + "_imp" for c in feats]
    return Pipeline(stages=[
        Imputer(inputCols=feats, outputCols=imp_cols, strategy="median"), # If missing value, inputer fills it with median.
        VectorAssembler(inputCols=imp_cols, outputCol="raw_features", handleInvalid="keep"), # Take the 8 features and make it into a single column.
        StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True), # Make features have comparable scale. Prevents variables with larger numerical magnitudes from dominating the linear-regression fitting process
        clf])

# List of candidates: Linear Regression, Random Forest, Gradient Boosted Trees.
CANDIDATES = []
for reg in [0.0, 0.01, 0.1]:
    CANDIDATES.append(("linreg", {"regParam": reg}, lambda reg=reg: LinearRegression(labelCol="label", featuresCol="features", regParam=reg, elasticNetParam=0.0, maxIter=100)))
for nt in [50, 100]:
    for md_ in [4, 6]:
        CANDIDATES.append(("random_forest", {"numTrees": nt, "maxDepth": md_},
            lambda nt=nt, md_=md_: RandomForestRegressor(labelCol="label", featuresCol="features", numTrees=nt, maxDepth=md_, seed=SEED)))
for it in [20, 50]:
    for md_ in [3, 5]:
        CANDIDATES.append(("gbt", {"maxIter": it, "maxDepth": md_},
            lambda it=it, md_=md_: GBTRegressor(labelCol="label", featuresCol="features", maxIter=it, maxDepth=md_, seed=SEED)))

# This runs the model and adds a "prediction" column with the predicted volatility for next week.
def score_df(model, df):
    return model.transform(df)

results, fitted = [], []
# For each candidate model:
# 1. Build the pipeline
# 2. Fit it on training data
# 3. Evaluate it on validation data
# 4. Save the metrics
# 5. Print the results
with stage("train_and_validate_all"):
    for name, params, mk in CANDIDATES:
        t0 = time.time()
        m = make_pipeline(mk()).fit(train)
        r = metrics(score_df(m, val))
        results.append({"model": name, "params": json.dumps(params), **{"val_" + k: v for k, v in r.items()}, "fit_s": round(time.time() - t0, 1)})
        fitted.append(m)
        print(results[-1])

# Print table sorted by validation RMSE (lowest first).
# The model with the lowest validation RMSE is chosen.
# For report.
spark.createDataFrame(results).orderBy(F.asc("val_rmse")).show(50, truncate=False)


{'model': 'linreg', 'params': '{"regParam": 0.0}', 'val_rmse': 0.010168, 'val_mae': 0.006615, 'val_r2': 0.2177, 'fit_s': 7.7}
{'model': 'linreg', 'params': '{"regParam": 0.01}', 'val_rmse': 0.010327, 'val_mae': 0.006609, 'val_r2': 0.193, 'fit_s': 3.3}
{'model': 'linreg', 'params': '{"regParam": 0.1}', 'val_rmse': 0.011095, 'val_mae': 0.006956, 'val_r2': 0.0685, 'fit_s': 3.8}
{'model': 'random_forest', 'params': '{"numTrees": 50, "maxDepth": 4}', 'val_rmse': 0.010378, 'val_mae': 0.006632, 'val_r2': 0.1849, 'fit_s': 8.6}
{'model': 'random_forest', 'params': '{"numTrees": 50, "maxDepth": 6}', 'val_rmse': 0.010301, 'val_mae': 0.006613, 'val_r2': 0.197, 'fit_s': 8.2}
{'model': 'random_forest', 'params': '{"numTrees": 100, "maxDepth": 4}', 'val_rmse': 0.010374, 'val_mae': 0.006638, 'val_r2': 0.1855, 'fit_s': 8.7}
{'model': 'random_forest', 'params': '{"numTrees": 100, "maxDepth": 6}', 'val_rmse': 0.010297, 'val_mae': 0.006619, 'val_r2': 0.1977, 'fit_s': 13.4}
{'model': 'gbt', 'params': '{"ma

In [ ]:
# ---- Extra (supporting comparison, NOT a candidate for model selection) ----
# Plain baseline:
#     next week's volatility = vol_12w
# (no learning, no training)

# Calibrated baseline:
#     train a Linear Regression using ONLY vol_12w as the feature.
# This shows whether the improvement comes from:
#   (1) learning a better scaling of vol_12w, or
#   (2) having the extra features (vol_1w, vol_4w, volume, etc.).

# If the calibrated baseline is already close to the full model, then most of the gain comes from rescaling vol_12w rather than from the additional features.
cal_model = make_pipeline(LinearRegression(labelCol="label", featuresCol="features", regParam=0.0, maxIter=100),
                          feats=["vol_12w"]).fit(train)
cal_val = metrics(score_df(cal_model, val))
best_val = min(results, key=lambda r: r["val_rmse"])
print("plain baseline (vol_12w as is)      VAL:", base_val["vol_12w"])
print("calibrated (linear on vol_12w only) VAL:", cal_val)
print("best model (all features)           VAL:", {k: v for k, v in best_val.items() if k.startswith("val_")})

plain baseline (vol_12w as is)      VAL: {'rmse': 0.011139, 'mae': 0.007658, 'r2': 0.0611}
calibrated (linear on vol_12w only) VAL: {'rmse': 0.010483, 'mae': 0.006693, 'r2': 0.1684}
best model (all features)           VAL: {'val_rmse': 0.010168, 'val_mae': 0.006615, 'val_r2': 0.2177}


## 4. Freeze the final model (chosen on validation RMSE) and evaluate ONCE on test
Do not change anything after this cell based on test results.

In [ ]:
# Find the best model (lowest validation RMSE).
best_i = min(range(len(results)), key=lambda i: results[i]["val_rmse"])
best, best_model = results[best_i], fitted[best_i]
print("FINAL:", best["model"], best["params"], "| validation:", {k: v for k, v in best.items() if k.startswith("val_")})

# Run the winning model on the test dataset and compute the metrics.
with stage("test_evaluation"):
    test_pred = score_df(best_model, test).cache()
    test_metrics = metrics(test_pred)
    base_test = {k: metrics(f(test)) for k, f in BASELINES.items()}     # <- Run the 12-week average baseline on the same test rows.
print("TEST  model   :", test_metrics); print("TEST  baselines:", base_test)
test_pred.select(F.avg("label").alias("avg_actual_vol"), F.avg("prediction").alias("avg_predicted_vol")).show()   # <- bias check: average actual vs average predicted volatility

FINAL: linreg {"regParam": 0.0} | validation: {'val_rmse': 0.010168, 'val_mae': 0.006615, 'val_r2': 0.2177}
[timing] test_evaluation: 0.77s
TEST  model   : {'rmse': 0.008882, 'mae': 0.005631, 'r2': 0.1005}
TEST  baselines: {'vol_12w': {'rmse': 0.009849, 'mae': 0.006395, 'r2': -0.106}}
+--------------------+--------------------+
|      avg_actual_vol|   avg_predicted_vol|
+--------------------+--------------------+
|0.011173448873773452|0.012309582272424293|
+--------------------+--------------------+



In [ ]:
# Same comparison on the test set.
#Run only after the final model is frozen; do not change any choice because of these numbers.
cal_test = metrics(score_df(cal_model, test))
print("plain baseline (vol_12w as is)      TEST:", base_test["vol_12w"])
print("calibrated (linear on vol_12w only) TEST:", cal_test)
print("final model (all features)          TEST:", test_metrics)

plain baseline (vol_12w as is)      TEST: {'rmse': 0.009849, 'mae': 0.006395, 'r2': -0.106}
calibrated (linear on vol_12w only) TEST: {'rmse': 0.008997, 'mae': 0.005726, 'r2': 0.077}
final model (all features)          TEST: {'rmse': 0.008882, 'mae': 0.005631, 'r2': 0.1005}


## 4b. OPTIONAL Part B: does news help? (Model B)
Skipped unless `NEWS_TABLE` is set in section 0. **Model A** = price features only (the main model, frozen above). **Model B** = the same model type and settings as Model A, plus `news_count` from Wikinews. Both are scored on the same 2017 test weeks.
Model A stays the main model: B is only a supporting comparison, and nothing is tuned on the test results. `news_count` is a market-wide weekly count (the same for every stock in a given week), so it can only help explain market-wide swings in volatility. State its coverage and availability limits in the report.

In [ ]:
news_compare = None
if NEWS_TABLE is not None:
    FEATS_B = FEATS + ["news_count"]
    with stage("model_B_news"):
        # Same model type and settings as the frozen Model A, so any difference comes from the news feature.
        clf_B = [mk for name, params, mk in CANDIDATES if name == best["model"] and json.dumps(params) == best["params"]][0]()
        model_B = make_pipeline(clf_B, FEATS_B).fit(train)
        pred_B = score_df(model_B, test).cache()
        news_compare = {"model_A_price_only": test_metrics, "model_B_price_plus_news": metrics(pred_B),
                        "baseline": base_test, "test_rows": pred_B.count()}
    print(json.dumps(news_compare, indent=2))
else:
    print("NEWS_TABLE not set: skipping optional Model B (news_count).")

NEWS_TABLE not set: skipping optional Model B (news_count).


## 5. Save the complete PipelineModel + reference predictions

In [ ]:
# Save the entire trained model so can load it later and save the test predictions.
# So later, can reload the model and run predictions without retraining.
with stage("save_model"):
    best_model.write().overwrite().save(MODEL_PATH)       # includes Imputer + Scaler + regressor
    (test_pred.select("symbol", "anchor_date", "prediction", "label")
              .write.mode("overwrite").parquet(REF_PRED_PATH)) # <- Saves the test predictions as the “answer key” that the reload check compares against.
print("model saved to", MODEL_PATH); print("reference predictions saved to", REF_PRED_PATH)

[timing] save_model: 2.14s
model saved to file:///content/task3_outputs/model_best
reference predictions saved to file:///content/task3_outputs/reference_test_predictions


## 5b. Turn predicted volatility into weights (weight = 1 / predicted volatility)
Riskier stocks (higher predicted volatility) get less weight. Two safeguards:
* **Floor**: a predicted volatility close to 0 would give an enormous weight, so predictions are floored at `VOL_FLOOR` (the `FLOOR_PCT` percentile of the *training* weekly volatilities, so nothing is learned from test).
* **Normalisation**: raw weights (1 / floored volatility) are divided by their sum within each week, so each week's weights add up to 1 (100%).

These weights are model diagnostics, not economic returns and not proof that a portfolio would have been investable.

In [ ]:
# Convert predicted volatility into portfolio weights:
# lower predicted volatility -> higher weight
# higher predicted volatility -> lower weight
#
# Then compare the actual volatility of:
# 1. the model-weighted portfolio
# 2. an equal-weight portfolio
#
# If the model-weighted portfolio has lower actual volatility,
# the weighting scheme is working as intended.

with stage("weights"):
    # Floor learned from the TRAINING weekly volatilities only.
    VOL_FLOOR = train.approxQuantile("label", [FLOOR_PCT], 0.001)[0]
    by_week = Window.partitionBy("anchor_date")
    weights = (test_pred.select("symbol", "anchor_date", "prediction", "label")
                 .withColumn("pred_vol_floored", F.greatest(F.col("prediction"), F.lit(VOL_FLOOR)))    # floor: never below VOL_FLOOR
                 .withColumn("raw_weight", 1.0 / F.col("pred_vol_floored"))                             # riskier stock -> smaller weight
                 .withColumn("weight", F.col("raw_weight") / F.sum("raw_weight").over(by_week))         # normalise: weights sum to 1 each week
                 .cache())
    weights.write.mode("overwrite").parquet(WEIGHTS_PATH)

    # For each week (anchor_date):
    # 1. Check that all stock weights add up to 1 (100% of the portfolio).
    # 2. Check the largest single-stock weight (to avoid extreme concentration).
    # 3. Calculate the actual volatility using the model's weights.
    # 4. Compare it with an equal-weight portfolio (every stock gets the same weight).
    #
    # If the model-weighted portfolio has lower actual volatility, the weighting scheme is working as intended.
    wk_check = weights.groupBy("anchor_date").agg(
        F.sum("weight").alias("w_sum"), F.max("weight").alias("w_max"), F.count("*").alias("n_stocks"),
        F.sum(F.col("weight") * F.col("label")).alias("weighted_actual_vol"), # Use the weights produced by the model.
        F.avg("label").alias("equal_weight_actual_vol"))  # Pretend every stock gets the same weight.

    weights_diag = wk_check.agg(
        F.min("w_sum").alias("min_weekly_weight_sum"), F.max("w_sum").alias("max_weekly_weight_sum"),
        F.max("w_max").alias("largest_single_weight"),
        F.avg("weighted_actual_vol").alias("avg_weighted_actual_vol"),
        F.avg("equal_weight_actual_vol").alias("avg_equal_weight_actual_vol")).first().asDict()
    weights_diag["vol_floor"] = VOL_FLOOR
    weights_diag["rows"] = weights.count()
    weights_diag["rows_floored"] = weights.filter(F.col("prediction") < VOL_FLOOR).count()   # how often the floor was needed
print(json.dumps(weights_diag, indent=2, default=str))
weights.orderBy(F.desc("weight")).show(10)   # <- the largest weights; check none are absurd

[timing] weights: 13.4s
{
  "min_weekly_weight_sum": 0.9999999999999981,
  "max_weekly_weight_sum": 1.0000000000000018,
  "largest_single_weight": 0.007202066145072726,
  "avg_weighted_actual_vol": 0.010387821642549719,
  "avg_equal_weight_actual_vol": 0.011173693348837438,
  "vol_floor": 0.003236542138,
  "rows": 25441,
  "rows_floored": 11
}
+------+-----------+--------------------+--------------+--------------------+------------------+--------------------+
|symbol|anchor_date|          prediction|         label|    pred_vol_floored|        raw_weight|              weight|
+------+-----------+--------------------+--------------+--------------------+------------------+--------------------+
|   ARE| 2017-03-24|0.001895419682847...|0.005474190113|      0.003236542138|308.97172270957776|0.007202066145072726|
|   RMD| 2017-07-28|7.213640366295539E-4| 0.03611231464|      0.003236542138|308.97172270957776|0.007029680385251...|
|  ANSS| 2017-06-23|0.002966131198747104| 0.00994723294|      0.

## 6. Evidence record (copy key numbers into report section 4 / Appendix A and B)

In [ ]:
evidence = {
    "run_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "spark_version": spark.version, "expected_spark_prefix": EXPECTED_SPARK_PREFIX, "spark_version_ok": SPARK_VERSION_OK,
    "python_version": sys.version.split()[0], "seed": SEED,
    "task": "predict next week's volatility (Task 2 target_next_vol: sample std of daily log returns from the anchor day's close to the day before the next anchor day) from the Task 2 weekly features",
    "splits": SPLITS, "features": FEATS,
    "task2_expected_rows": TASK2_EXPECTED_ROWS, "loaded_rows": loaded_counts,
    "input_tables": [TRAIN_TABLE, VAL_TABLE, TEST_TABLE] if WEEKLY_DIR is None else WEEKLY_DIR,
    "row_counts": {p: ok.filter(F.col("period") == p).count() for p in ["train", "val", "test"]},
    "exclusions": {f"{r['period']}|{r['status']}": r["count"] for r in summary.collect()},
    "baseline": "volatility over the previous 12 weeks (vol_12w from Task 2)", "validation_baselines": base_val, "test_baselines": base_test,
    "calibrated_baseline": {"description": "Linear regression trained using only vol_12w", "validation": cal_val, "test": cal_test},
    "weights": {"rule": "1 / max(predicted vol, vol_floor), normalised to sum to 1 within each week", "floor_pct_of_train": FLOOR_PCT, **weights_diag},
    "models": results, "final_model": {"model": best["model"], "params": best["params"]},
    "validation_final": {k: v for k, v in best.items() if k.startswith("val_")},
    "test_model": test_metrics,
    "news_comparison": news_compare, "model_path": MODEL_PATH, "weights_path": WEIGHTS_PATH, "timings_s": TIMINGS}
save_json(evidence, EVIDENCE_PATH + "/task3_run_evidence")
print(json.dumps(evidence, indent=2, default=str))

{
  "run_timestamp_utc": "2026-11-06T10:05:40.041972+00:00",
  "spark_version": "4.0.5",
  "expected_spark_prefix": "3.5",
  "spark_version_ok": false,
  "python_version": "3.13.18",
  "seed": 42,
  "task": "predict next week's volatility (Task 2 target_next_vol: sample std of daily log returns from the anchor day's close to the day before the next anchor day) from the Task 2 weekly features",
  "splits": {
    "train": [
      "2013-01-01",
      "2015-12-31"
    ],
    "val": [
      "2016-01-01",
      "2016-12-31"
    ],
    "test": [
      "2017-01-01",
      "2017-12-31"
    ]
  },
  "features": [
    "vol_1w",
    "vol_4w",
    "vol_12w",
    "vol_52w",
    "high_low_range",
    "weekly_return",
    "volume_vs_normal",
    "split_flag"
  ],
  "task2_expected_rows": {
    "train": 66907,
    "val": 25310,
    "test": 25441
  },
  "loaded_rows": {
    "train": 66907,
    "test": 25441,
    "val": 25310
  },
  "input_tables": "task2_outputs",
  "row_counts": {
    "train": 66907,
 

## 7. FRESH-SESSION RELOAD CHECK (no retraining)
Restart the kernel / open a new session, run **section 0 only**, then run this cell. It loads the saved model, rescoring the stored test features, and compares against the reference predictions within `TOL`.

In [ ]:
fresh = SparkSession.builder.getOrCreate()

# Which Spark version saved the model? (A different major.minor version may be incompatible.)
saved_spark_version = json.loads(spark.read.text(MODEL_PATH + "/metadata").first()[0]).get("sparkVersion")
print("model saved with Spark", saved_spark_version, "| loading with Spark", spark.version)
if saved_spark_version and saved_spark_version.split(".")[:2] != spark.version.split(".")[:2]:
    print("WARNING: Spark major.minor version differs from the one that saved the model; loading may fail or give different results.")

# Load the model without retraining.
model2 = PipelineModel.load(MODEL_PATH)                       # fails loudly if files are missing/incompatible

# Run the reloaded model on the same test rows and compute new predicted volatility (p_new).
feat = spark.read.parquet(FEATURES_PATH).filter("status = 'ok' AND period = 'test'")
new = (model2.transform(feat)
             .select("symbol", "anchor_date", F.col("prediction").alias("p_new")))

# Load the original predictions (p_ref).
ref = spark.read.parquet(REF_PRED_PATH).select("symbol", "anchor_date", F.col("prediction").alias("p_ref"))

# Join the new predictions and the reference predictions by stock and date.
cmp = new.join(ref, ["symbol", "anchor_date"], "full_outer")

# Compute comparison statistics.
# n: Total number of rows compared.
# unmatched_rows: detects missing or extra rows.
# max_abs_diff: checks numerical consistency.
st = cmp.agg(F.count("*").alias("n"),
             F.sum(F.when(F.col("p_new").isNull() | F.col("p_ref").isNull(), 1).otherwise(0)).alias("unmatched_rows"),
             F.max(F.abs(F.col("p_new") - F.col("p_ref"))).alias("max_abs_diff")).first()

# Pass or Fail
# If No missing rows, Predicted volatilities differ by at most a tiny tolerance = PASS
# Otherwise FAIL
passed = (st["unmatched_rows"] == 0) and (st["max_abs_diff"] is not None) and (st["max_abs_diff"] <= TOL)
record = {"check": "reload saved PipelineModel in fresh session and reproduce test predictions",
          "command": "PipelineModel.load(MODEL_PATH).transform(stored test features)", "tolerance": TOL,
          "rows_compared": st["n"], "unmatched_rows": st["unmatched_rows"], "max_abs_pred_diff": st["max_abs_diff"],
          "result": "PASS" if passed else "FAIL",
          "spark_version": spark.version, "spark_version_saved": saved_spark_version,
          "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "model_path": MODEL_PATH}
save_json(record, EVIDENCE_PATH + "/task3_reload_check")
print(json.dumps(record, indent=2, default=str))

model saved with Spark 4.0.5 | loading with Spark 4.0.5
{
  "check": "reload saved PipelineModel in fresh session and reproduce test predictions",
  "command": "PipelineModel.load(MODEL_PATH).transform(stored test features)",
  "tolerance": 1e-09,
  "rows_compared": 25441,
  "unmatched_rows": 0,
  "max_abs_pred_diff": 0.0,
  "result": "PASS",
  "spark_version": "4.0.5",
  "spark_version_saved": "4.0.5",
  "timestamp_utc": "2026-11-06T10:06:49.638891+00:00",
  "model_path": "file:///content/task3_outputs/model_best"
}
